# PrivAware v2 — Day 2: Train Phishing Detection Model

**Goal:** Fine-tune DistilBERT on your saved phishing dataset. End of today you have a trained model with accuracy and F1 score to show the panel.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** → Save
- Run each cell one by one (Shift+Enter)
- Cell 7 (training) takes 40–50 minutes — just leave the tab open

**What you need from Day 1:** The phishing_train.json, phishing_val.json, phishing_test.json files in your Google Drive under privaware-v2/

## Cell 1 — Install libraries

In [ ]:
!pip install transformers datasets torch scikit-learn -q
print('All libraries installed!')

## Cell 2 — Mount Drive and load saved data

In [ ]:
from google.colab import drive
import json

drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/privaware-v2/'

def load_split(name):
    with open(f'{SAVE_PATH}phishing_{name}.json', 'r') as f:
        data = json.load(f)
    texts = [d['text'] for d in data]
    labels = [d['label'] for d in data]
    return texts, labels

X_train, y_train = load_split('train')
X_val, y_val = load_split('val')
X_test, y_test = load_split('test')

print(f'Train: {len(X_train)} samples')
print(f'Val:   {len(X_val)} samples')
print(f'Test:  {len(X_test)} samples')
print('Data loaded successfully!')

## Cell 3 — Tokenize the data
DistilBERT needs text converted to numbers. This is called tokenization. Takes 2-3 minutes.

In [ ]:
from transformers import DistilBertTokenizerFast
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, labels, max_length=128):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt'
    )
    return encodings, torch.tensor(labels)

print('Tokenizing training set...')
train_encodings, train_labels = tokenize(X_train, y_train)

print('Tokenizing validation set...')
val_encodings, val_labels = tokenize(X_val, y_val)

print('Tokenizing test set...')
test_encodings, test_labels = tokenize(X_test, y_test)

print('Tokenization complete!')
print(f'Input shape: {train_encodings["input_ids"].shape}')

## Cell 4 — Create PyTorch dataset objects

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PhishingDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = PhishingDataset(train_encodings, train_labels)
val_dataset   = PhishingDataset(val_encodings, val_labels)
test_dataset  = PhishingDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print('Datasets ready!')

## Cell 5 — Load DistilBERT model
This downloads the pre-trained DistilBERT and adds a classification layer on top. That classification layer is what we train.

In [ ]:
from transformers import DistilBertForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if str(device) == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime -> Change Runtime Type -> T4 GPU')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)

print('Model loaded!')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## Cell 6 — Set up optimizer and scheduler

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

NUM_EPOCHS = 3
num_training_steps = NUM_EPOCHS * len(train_loader)

lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=100,
    num_training_steps=num_training_steps
)

print(f'Training for {NUM_EPOCHS} epochs')
print(f'Total steps: {num_training_steps}')
print(f'Estimated time on T4 GPU: ~40-50 minutes')
print('Ready to train!')

## Cell 7 — TRAIN THE MODEL

**This takes 40–50 minutes. Do not close the tab.**

You will see progress printed every 100 steps. After each epoch you will see Val Acc and Val F1.
The best model is automatically saved whenever validation accuracy improves.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import time

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)
    return acc, f1

print('Starting training...')
print('=' * 55)

best_val_acc = 0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 100 == 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f} | Elapsed: {time.time()-start:.0f}s')

    val_acc, val_f1 = evaluate(val_loader)
    avg_loss = total_loss / len(train_loader)

    history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})

    print(f'\nEpoch {epoch+1} Summary:')
    print(f'  Avg Loss : {avg_loss:.4f}')
    print(f'  Val Acc  : {val_acc*100:.2f}%')
    print(f'  Val F1   : {val_f1:.4f}')
    print(f'  Time     : {time.time()-start:.0f}s')
    print('=' * 55)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('/content/best_model')
        tokenizer.save_pretrained('/content/best_model')
        print(f'  *** Best model saved (Val Acc: {val_acc*100:.2f}%) ***')

print('\nTraining complete!')

## Cell 8 — Evaluate on test set and get final numbers

In [ ]:
from sklearn.metrics import classification_report
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

# Load the best saved model
model     = DistilBertForSequenceClassification.from_pretrained('/content/best_model')
tokenizer = DistilBertTokenizerFast.from_pretrained('/content/best_model')
model.to(device)

test_acc, test_f1 = evaluate(test_loader)

print('=' * 55)
print('FINAL RESULTS — WRITE THESE DOWN FOR YOUR PANEL')
print('=' * 55)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test F1 Score : {test_f1:.4f}')
print('=' * 55)

# Detailed per-class report
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print('\nDetailed Classification Report:')
print(classification_report(all_labels, all_preds, target_names=['Legitimate', 'Phishing']))

## Cell 9 — Plot training curves
Save these charts — they look great in your panel presentation.

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
losses = [h['loss']  for h in history]
accs   = [h['val_acc'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, losses, 'b-o', linewidth=2, markersize=8)
axes[0].set_title('Training Loss per Epoch', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in accs], 'g-o', linewidth=2, markersize=8)
axes[1].set_title('Validation Accuracy per Epoch', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim([80, 100])
axes[1].grid(True, alpha=0.3)

plt.suptitle('PrivAware — Phishing Detection Model Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

## Cell 10 — Save everything to Google Drive

In [ ]:
import shutil, os

DRIVE_MODEL_PATH = '/content/drive/MyDrive/privaware-v2/phishing_model/'
os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)

# Save model
shutil.copytree('/content/best_model', DRIVE_MODEL_PATH, dirs_exist_ok=True)
print(f'Model saved to: {DRIVE_MODEL_PATH}')

# Save training curves
shutil.copy('/content/training_curves.png', SAVE_PATH + 'phishing_training_curves.png')
print('Training curves saved to Drive.')

print('\nFiles in model folder:')
for f in os.listdir(DRIVE_MODEL_PATH):
    print(f'  {f}')

print('\n' + '='*55)
print('DAY 2 COMPLETE!')
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test F1 Score : {test_f1:.4f}')
print('Write these numbers in your master plan document.')
print('Message me for Day 3.')
print('='*55)